# Project 1: Thermal Sensation — Day 1: Load and clean the data

Today we will load the ASHRAE files, inspect their basic structure, and prepare a clean table for the **thermal sensation** project.

We will keep the original CSV files unchanged. Cleaning means fixing clear formatting issues and making a target-specific copy; it does **not** mean deleting every row with any missing value.

## 1. Import pandas and find the files

`Path` helps us build file paths. `pandas` reads CSV files into DataFrames (tables). The cells work when the notebook is opened from the project folder or its `notebooks` folder.

In [ ]:
from pathlib import Path

import pandas as pd

# Find the project folder, even if the notebook was opened from notebooks/.
project_folder = Path.cwd()
if not (project_folder / "data" / "raw").exists():
    project_folder = project_folder.parent

raw_data_folder = project_folder / "data" / "raw"
measurements_path = raw_data_folder / "Ashrae_Comfort_Cooling_Database_v2.1.0.csv"
metadata_path = raw_data_folder / "Ashrae_Comfort_Cooling_Database_Metadata.csv"

# Give a clear error if a file was moved or renamed.
if not measurements_path.exists():
    raise FileNotFoundError(f"Measurements file not found: {measurements_path}")

if not metadata_path.exists():
    raise FileNotFoundError(f"Metadata file not found: {metadata_path}")

print("Both data files were found.")

## 2. Load the measurements and metadata

`building_id` is an identifier, not a number to calculate with, so we load it as text in both files. We also load `subject_id` as text because it is an ID. The three response/ID columns below are text labels and can otherwise cause pandas' mixed-type warning.

In [ ]:
measurement_text_columns = {
    "building_id": "string",
    "subject_id": "string",
    "air_movement_acceptability": "string",
    "air_movement_preference": "string",
}

metadata_text_columns = {
    "building_id": "string",
}

# Pandas recognizes the dataset's NA markers as missing values by default.
measurements = pd.read_csv(measurements_path, dtype=measurement_text_columns)
metadata = pd.read_csv(metadata_path, dtype=metadata_text_columns)

print("Measurements:", measurements.shape[0], "rows and", measurements.shape[1], "columns")
print("Metadata:", metadata.shape[0], "rows and", metadata.shape[1], "columns")

## 3. Take a first look

Before changing anything, inspect a few rows, the column types, and the thermal-sensation target. This helps us understand what needs cleaning and avoids changing data just because it looks unfamiliar.

In [ ]:
display(measurements.head())
display(metadata.head())

print("Measurement data types:")
display(measurements.dtypes.to_frame(name="data_type"))

print("Thermal sensation values, including missing values:")
display(measurements["thermal_sensation"].value_counts(dropna=False).head(20))

print("Missing thermal-sensation targets:", measurements["thermal_sensation"].isna().sum())

## 4. Clean text fields and the target

We will remove accidental spaces around text values and make sure the thermal-sensation target is numeric. This notebook prepares Project 1 only; we will prepare the preference target separately when Project 2 begins.

Any non-empty sensation value that cannot be converted to a number becomes missing. We count those conversions so unexpected values are visible instead of silently ignored.

In [ ]:
# Remove leading and trailing spaces from text columns.
for column in measurements.select_dtypes(include=["object", "string"]).columns:
    measurements[column] = measurements[column].str.strip()

for column in metadata.select_dtypes(include=["object", "string"]).columns:
    metadata[column] = metadata[column].str.strip()

# Convert sensation votes to numbers. Invalid non-empty values are counted below.
sensation_before_conversion = measurements["thermal_sensation"]
sensation_after_conversion = pd.to_numeric(sensation_before_conversion, errors="coerce")
invalid_sensation_count = (
    sensation_before_conversion.notna() & sensation_after_conversion.isna()
).sum()
measurements["thermal_sensation"] = sensation_after_conversion

print("Invalid non-empty sensation values converted to missing:", invalid_sensation_count)
print("Sensation values range:", measurements["thermal_sensation"].min(), "to",
      measurements["thermal_sensation"].max())

## 5. Check IDs and duplicate records

The database has a `record_id` to identify measurement rows. Repeated measurement values do not always mean a mistake: different people can experience the same conditions. We will report duplicate counts but will not delete rows based only on matching values.

In [ ]:
print("Are all record_id values unique?", measurements["record_id"].is_unique)
print("Fully identical rows:", measurements.duplicated().sum())

# Ignore row identifiers for this check to find repeated measurement payloads.
measurement_values = measurements.drop(columns=["index", "record_id"])
print("Repeated rows apart from index and record_id:", measurement_values.duplicated().sum())

print("Missing building IDs in measurements:", measurements["building_id"].isna().sum())
print("Missing building IDs in metadata:", metadata["building_id"].isna().sum())

## 6. Join building metadata to measurements

The metadata adds global context such as country, climate, building type, and cooling type. We join on `building_id`.

The metadata should have one row per building. We use `validate="many_to_one"` so pandas checks this relationship, and then confirm the join kept the original measurement row count.

In [ ]:
if not metadata["building_id"].is_unique:
    raise ValueError("Metadata contains repeated building_id values; review before joining.")

joined_data = measurements.merge(
    metadata,
    on="building_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print("Rows before joining:", len(measurements))
print("Rows after joining:", len(joined_data))
print("Join results:")
display(joined_data["_merge"].value_counts())

if len(joined_data) != len(measurements):
    raise ValueError("The metadata join changed the number of measurement rows.")

if (joined_data["_merge"] != "both").any():
    raise ValueError("Some measurement rows did not find matching building metadata.")

# The indicator was only needed to check the join, so remove it now.
joined_data = joined_data.drop(columns="_merge")
print("Join check passed.")

## 7. Make the cleaned thermal-sensation dataset

A supervised model needs a known target value, so rows without `thermal_sensation` cannot be used to train or evaluate this target. We will remove only those rows from this **project-specific copy**.

We will not remove rows just because an input feature is missing. Day 2 will help us choose features; missing feature values can be handled later in a model pipeline. We also keep repeated measurement payloads unless we find evidence they are erroneous.

In [ ]:
rows_before_target_check = len(joined_data)

# Keep rows that have a usable thermal-sensation target.
cleaned_data = joined_data.loc[
    joined_data["thermal_sensation"].notna()
].copy()

print("Rows before removing missing targets:", rows_before_target_check)
print("Rows removed because sensation target is missing:",
      rows_before_target_check - len(cleaned_data))
print("Rows in cleaned thermal-sensation dataset:", len(cleaned_data))
print("Remaining missing sensation targets:", cleaned_data["thermal_sensation"].isna().sum())

# The documented vote range is -3 to +3. Check values without rounding them.
outside_documented_range = ~cleaned_data["thermal_sensation"].between(-3, 3)
print("Sensation targets outside -3 to +3:", outside_documented_range.sum())

## 8. Save the cleaned copy

The raw files stay unchanged in `data/raw/`. We save the joined table with a valid sensation target in `data/processed/` so the next notebook can use it. If you rerun this notebook, it will recreate this processed file.

In [ ]:
processed_data_folder = project_folder / "data" / "processed"
processed_data_folder.mkdir(parents=True, exist_ok=True)

cleaned_data_path = processed_data_folder / "thermal_sensation_cleaned.csv"
cleaned_data.to_csv(cleaned_data_path, index=False)

print("Cleaned dataset saved to:", cleaned_data_path)

## Day 1 summary

- We loaded the measurements and metadata as separate tables and joined them using `building_id`.
- We standardized whitespace and checked that `thermal_sensation` is numeric. Preparation here is specific to Project 1.
- We removed only records without the thermal-sensation target from the project-specific cleaned copy. We did not drop rows for missing input features, automatically remove repeated measurements, or alter the raw CSVs.
- The cleaned output is `data/processed/thermal_sensation_cleaned.csv`.

Next, Day 2 will explore this cleaned data and choose candidate features. Any imputation or other learned preprocessing will happen later, after we define the training and test groups.